# 01 数据清洗与规范化

本 Notebook 输出召回、特征工程与建模使用的规范化 Parquet 数据，并生成 `data_quality_report.json`。探索性可视化由 `02_data_analysis_visualization.ipynb` 完成。重复交易会被保留，因为同一顾客同一天购买同一商品可能代表多件商品。

In [ ]:
import json, os, shutil, sys, time
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
pd.set_option('display.max_columns', 80)
pd.set_option('display.float_format', lambda value: f'{value:.6f}')
REQUIRED_FILES = {'customers.csv', 'articles.csv', 'transactions_train.csv', 'sample_submission.csv'}
def find_data_dir():
    candidates = [Path.cwd()/'data', Path.cwd().parent/'data', Path.cwd().parents[1]/'data', Path('/kaggle/input/h-and-m-personalized-fashion-recommendations')]
    for candidate in candidates:
        if candidate.is_dir() and REQUIRED_FILES.issubset({path.name for path in candidate.iterdir()}):
            return candidate.resolve()
    raise FileNotFoundError(f'未找到包含 {sorted(REQUIRED_FILES)} 的数据目录')
DATA_DIR = find_data_dir()
if Path('/kaggle/working').exists():
    OUTPUT_DIR = Path('/kaggle/working/hm_recsys_clean')
else:
    OUTPUT_DIR = next((base / 'optimization' for base in (Path.cwd(), *Path.cwd().parents) if (base / 'optimization' / 'code').is_dir()), Path.cwd())/'output'/'clean'
TEMP_DIR = OUTPUT_DIR.parent/(OUTPUT_DIR.name + '.tmp')
if TEMP_DIR.exists():
    shutil.rmtree(TEMP_DIR)
TEMP_DIR.mkdir(parents=True, exist_ok=True)
with (DATA_DIR/'articles.csv').open('rb') as article_file:
    ARTICLES_COMPRESSION = 'zip' if article_file.read(4) == b'PK\x03\x04' else None
print('Python:', sys.version.split()[0])
print('pandas:', pd.__version__)
print('pyarrow:', pa.__version__)
print('DATA_DIR:', DATA_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('ARTICLES_COMPRESSION:', ARTICLES_COMPRESSION or 'none')

In [ ]:
def normalize_article_ids(article_ids):
    normalized = article_ids.astype('string').str.strip().str.zfill(10)
    malformed = normalized.isna() | ~normalized.str.fullmatch(r'\d{10}', na=False)
    if malformed.any():
        raise ValueError(f'非法 article_id 示例: {article_ids.loc[malformed].head().tolist()}')
    return normalized.astype(str), int(malformed.sum())
def normalize_customer_ids(customer_ids):
    text_ids = customer_ids.astype('string').str.strip()
    malformed = text_ids.isna() | ~text_ids.str.fullmatch(r'[0-9a-f]{64}', na=False)
    if malformed.any():
        raise ValueError(f'非法 customer_id 示例: {customer_ids.loc[malformed].head().tolist()}')
    keys = np.fromiter((int(value[-16:], 16) for value in text_ids), dtype=np.uint64, count=len(text_ids))
    return text_ids.astype(str), keys, int(malformed.sum())
def categorical(series, fill_value='UNKNOWN'):
    return series.astype('string').str.strip().str.upper().fillna(fill_value).replace({'': fill_value}).astype('category')
def nullable_numeric(series, dtype):
    return pd.to_numeric(series, errors='coerce').astype(dtype)
def null_counts(frame):
    return {column: int(value) for column, value in frame.isna().sum().items()}
def write_json(path, payload):
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
print('规范化工具函数已定义。')

In [ ]:
customers_path = DATA_DIR/'customers.csv'
raw_customers = pd.read_csv(customers_path, dtype={'customer_id': 'string'})
customer_ids, customer_keys, malformed_customers = normalize_customer_ids(raw_customers['customer_id'])
customers_clean = pd.DataFrame({
    'customer_id': customer_ids,
    'customer_key': customer_keys,
    'fn_flag': nullable_numeric(raw_customers['FN'], 'boolean'),
    'active_flag': nullable_numeric(raw_customers['Active'], 'boolean'),
    'club_member_status': categorical(raw_customers['club_member_status']),
    'fashion_news_frequency': categorical(raw_customers['fashion_news_frequency']),
    'age': nullable_numeric(raw_customers['age'], 'Int16'),
    'postal_code': raw_customers['postal_code'].astype('string').fillna('UNKNOWN').astype(str),
})
duplicate_customer_rows = int(raw_customers['customer_id'].duplicated().sum())
duplicate_customer_keys = int(customers_clean['customer_key'].duplicated().sum())
if duplicate_customer_rows or duplicate_customer_keys:
    raise ValueError(f'顾客 ID 或压缩键重复: {duplicate_customer_rows}, {duplicate_customer_keys}')
customers_clean.to_parquet(TEMP_DIR/'customers_clean.parquet', index=False, compression='zstd')
customer_report = {
    'rows': int(len(raw_customers)),
    'columns': list(raw_customers.columns),
    'source_null_counts': null_counts(raw_customers),
    'malformed_customer_ids': malformed_customers,
    'duplicate_customer_ids': duplicate_customer_rows,
    'customer_key_collisions': duplicate_customer_keys,
    'clean_null_counts': null_counts(customers_clean),
    'age_min': int(customers_clean['age'].min()),
    'age_max': int(customers_clean['age'].max()),
    'unknown_age_rows': int(customers_clean['age'].isna().sum()),
}
print(customers_clean.dtypes)
display(customers_clean.head())
print(json.dumps(customer_report, ensure_ascii=False, indent=2))

In [ ]:
articles_path = DATA_DIR/'articles.csv'
raw_articles = pd.read_csv(articles_path, compression=ARTICLES_COMPRESSION, dtype={'article_id': 'string', 'product_code': 'int32'})
article_ids, malformed_articles = normalize_article_ids(raw_articles['article_id'])
articles_clean = raw_articles.copy()
articles_clean['article_id'] = article_ids
articles_clean['product_code'] = raw_articles['product_code'].astype('int32')
articles_clean['prefix_product_code'] = article_ids.str.lstrip('0').str[:6].astype('int32')
category_columns = [
    'product_type_name', 'product_group_name', 'graphical_appearance_name',
    'colour_group_name', 'perceived_colour_value_name', 'perceived_colour_master_name',
    'department_name', 'index_code', 'index_name', 'index_group_name',
    'section_name', 'garment_group_name'
]
for column in category_columns:
    articles_clean[column] = categorical(articles_clean[column])
articles_clean['prod_name'] = articles_clean['prod_name'].astype('string').str.strip().fillna('')
articles_clean['detail_desc'] = articles_clean['detail_desc'].astype('string').str.strip().fillna('')
duplicate_article_rows = int(raw_articles['article_id'].duplicated().sum())
if duplicate_article_rows:
    raise ValueError(f'article_id 重复: {duplicate_article_rows}')
if articles_clean.isna().any().any():
    raise ValueError(f'商品清洗后仍存在缺失值: {null_counts(articles_clean)}')
articles_clean.to_parquet(TEMP_DIR/'articles_clean.parquet', index=False, compression='zstd')
article_report = {
    'rows': int(len(raw_articles)),
    'columns': list(raw_articles.columns),
    'source_null_counts': null_counts(raw_articles),
    'malformed_article_ids': malformed_articles,
    'duplicate_article_ids': duplicate_article_rows,
    'clean_null_counts': null_counts(articles_clean),
    'product_code_prefix_mismatches': int((articles_clean['product_code'] != articles_clean['prefix_product_code']).sum()),
}
print(articles_clean.dtypes)
display(articles_clean.head())
print(json.dumps(article_report, ensure_ascii=False, indent=2))

In [ ]:
transactions_path = DATA_DIR/'transactions_train.csv'
transaction_schema = pa.schema([
    pa.field('t_dat', pa.date32(), nullable=False),
    pa.field('customer_key', pa.uint64(), nullable=False),
    pa.field('article_id', pa.string(), nullable=False),
    pa.field('price', pa.float64(), nullable=False),
    pa.field('sales_channel_id', pa.int8(), nullable=False),
])
customer_key_universe = np.sort(customers_clean['customer_key'].to_numpy(dtype=np.uint64))
article_number_universe = np.sort(articles_clean['article_id'].astype('int32').to_numpy())
transaction_writer = None
transaction_metrics = {
    'source_null_counts': {column: 0 for column in ['t_dat', 'customer_id', 'article_id', 'price', 'sales_channel_id']},
    'malformed_customer_ids': 0,
    'malformed_article_ids': 0,
    'invalid_dates': 0,
    'invalid_prices': 0,
    'invalid_sales_channels': 0,
    'unknown_customers': 0,
    'unknown_articles': 0,
    'duplicate_like_rows': 0,
    'rows': 0,
    'chunk_summaries': [],
}
transaction_started = time.time()
try:
    for chunk_number, chunk in enumerate(pd.read_csv(
        transactions_path,
        chunksize=2_000_000,
        dtype={'customer_id': 'string', 'article_id': 'string'},
    ), start=1):
        for column, value in null_counts(chunk).items():
            transaction_metrics['source_null_counts'][column] += value
        chunk_customer_ids, chunk_customer_keys, malformed_customers = normalize_customer_ids(chunk['customer_id'])
        chunk_article_ids, malformed_articles = normalize_article_ids(chunk['article_id'])
        transaction_dates = pd.to_datetime(chunk['t_dat'], format='%Y-%m-%d', errors='coerce')
        prices = pd.to_numeric(chunk['price'], errors='coerce')
        sales_channels = pd.to_numeric(chunk['sales_channel_id'], errors='coerce')
        article_numbers = chunk_article_ids.astype('int32').to_numpy()
        unknown_customer_mask = ~np.isin(chunk_customer_keys, customer_key_universe)
        unknown_article_mask = ~np.isin(article_numbers, article_number_universe)
        invalid_date_mask = transaction_dates.isna()
        invalid_price_mask = prices.isna() | ~np.isfinite(prices.to_numpy()) | (prices < 0)
        invalid_channel_mask = ~sales_channels.isin([1, 2])
        duplicate_like_mask = chunk.duplicated(keep=False)
        validation_counts = {
            'malformed_customer_ids': malformed_customers,
            'malformed_article_ids': malformed_articles,
            'invalid_dates': int(invalid_date_mask.sum()),
            'invalid_prices': int(invalid_price_mask.sum()),
            'invalid_sales_channels': int(invalid_channel_mask.sum()),
            'unknown_customers': int(unknown_customer_mask.sum()),
            'unknown_articles': int(unknown_article_mask.sum()),
        }
        if any(validation_counts.values()):
            raise ValueError(f'第 {chunk_number} 个交易分块未通过校验: {validation_counts}')
        clean_chunk = pd.DataFrame({
            't_dat': transaction_dates.dt.date,
            'customer_key': chunk_customer_keys,
            'article_id': chunk_article_ids,
            'price': prices.astype('float64'),
            'sales_channel_id': sales_channels.astype('int8'),
        })
        transaction_table = pa.Table.from_pandas(clean_chunk, schema=transaction_schema, preserve_index=False, safe=True)
        if transaction_writer is None:
            transaction_writer = pq.ParquetWriter(TEMP_DIR/'transactions_clean.parquet', transaction_schema, compression='zstd')
        transaction_writer.write_table(transaction_table)
        transaction_metrics['rows'] += len(chunk)
        transaction_metrics['duplicate_like_rows'] += int(duplicate_like_mask.sum())
        for key, value in validation_counts.items():
            transaction_metrics[key] += value
        transaction_metrics['chunk_summaries'].append({
            'chunk': chunk_number,
            'rows': int(len(chunk)),
            'date_min': str(transaction_dates.min().date()),
            'date_max': str(transaction_dates.max().date()),
            'price_min': float(prices.min()),
            'price_max': float(prices.max()),
        })
        print(f'交易分块 {chunk_number:02d}: {len(chunk):,} 行')
finally:
    if transaction_writer is not None:
        transaction_writer.close()
transaction_report = {
    'rows': transaction_metrics['rows'],
    'columns': ['t_dat', 'customer_id', 'article_id', 'price', 'sales_channel_id'],
    'source_null_counts': transaction_metrics['source_null_counts'],
    **{key: value for key, value in transaction_metrics.items() if key not in {'source_null_counts', 'chunk_summaries', 'rows'}},
    'date_min': min(item['date_min'] for item in transaction_metrics['chunk_summaries']),
    'date_max': max(item['date_max'] for item in transaction_metrics['chunk_summaries']),
    'price_min': min(item['price_min'] for item in transaction_metrics['chunk_summaries']),
    'price_max': max(item['price_max'] for item in transaction_metrics['chunk_summaries']),
    'elapsed_seconds': round(time.time() - transaction_started, 3),
    'decision': '重复交易全部保留；重复行可能表示同一天购买多件相同商品。',
}
write_json(TEMP_DIR/'transaction_chunk_summaries.json', transaction_metrics['chunk_summaries'])
print(json.dumps(transaction_report, ensure_ascii=False, indent=2))

In [ ]:
submission_path = DATA_DIR/'sample_submission.csv'
raw_submission = pd.read_csv(submission_path, dtype={'customer_id': 'string', 'prediction': 'string'})
submission_ids, submission_keys, malformed_submission_customers = normalize_customer_ids(raw_submission['customer_id'])
submission_customers = pd.DataFrame({'customer_id': submission_ids, 'customer_key': submission_keys})
duplicate_submission_customers = int(raw_submission['customer_id'].duplicated().sum())
unknown_submission_customers = int((~np.isin(submission_keys, customer_key_universe)).sum())
if duplicate_submission_customers or unknown_submission_customers:
    raise ValueError(f'提交顾客重复或未知: {duplicate_submission_customers}, {unknown_submission_customers}')
article_id_set = set(articles_clean['article_id'])
prediction_lists = raw_submission['prediction'].fillna('').str.split()
invalid_prediction_rows = int(prediction_lists.map(lambda items: len(items) != 12 or len(items) != len(set(items)) or any((not item.isdigit()) or len(item) != 10 for item in items)).sum())
unknown_prediction_articles = int(prediction_lists.map(lambda items: any(item not in article_id_set for item in items)).sum())
submission_customers.to_parquet(TEMP_DIR/'submission_customers.parquet', index=False, compression='zstd')
submission_report = {
    'rows': int(len(raw_submission)),
    'columns': list(raw_submission.columns),
    'malformed_customer_ids': malformed_submission_customers,
    'duplicate_customer_ids': duplicate_submission_customers,
    'unknown_customers': unknown_submission_customers,
    'invalid_sample_prediction_rows': invalid_prediction_rows,
    'sample_predictions_with_unknown_articles': unknown_prediction_articles,
    'note': 'sample_submission 的 prediction 仅用于格式检查，不作为模型训练标签。',
}
print(submission_report)

In [ ]:
artifact_names = ['customers_clean.parquet', 'articles_clean.parquet', 'transactions_clean.parquet', 'submission_customers.parquet', 'data_quality_report.json', 'transaction_chunk_summaries.json']
quality_report = {
    'generated_at': time.strftime('%Y-%m-%d %H:%M:%S %z'),
    'environment': {
        'python': sys.version.split()[0],
        'pandas': pd.__version__,
        'pyarrow': pa.__version__,
        'articles_source_compression': ARTICLES_COMPRESSION or 'none',
    },
    'data_dir': str(DATA_DIR),
    'customers': customer_report,
    'articles': article_report,
    'transactions': transaction_report,
    'submission': submission_report,
    'normalization_decisions': [
        'customer_key 使用 customer_id 后 16 位十六进制转换得到的 uint64，已检查碰撞。',
        'article_id 统一为 10 位字符串，交易与商品表均可无损互相匹配。',
        '顾客类别缺失值统一为 UNKNOWN，年龄保留可空整数，不主动填充。',
        '商品类别缺失值统一为 UNKNOWN，文本描述缺失值统一为空字符串。',
        '交易价格保留 float64，日期为 date32，渠道为 int8。',
        '交易重复行保留，因为其可能代表同一天购买多件相同商品。',
    ],
    'artifacts': {name: int((TEMP_DIR/name).stat().st_size) for name in artifact_names if (TEMP_DIR/name).exists()},
}
write_json(TEMP_DIR/'data_quality_report.json', quality_report)
quality_report['artifacts']['data_quality_report.json'] = int((TEMP_DIR/'data_quality_report.json').stat().st_size)
write_json(TEMP_DIR/'data_quality_report.json', quality_report)
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
TEMP_DIR.rename(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
summary = {
    'customers': customer_report['rows'],
    'articles': article_report['rows'],
    'transactions': transaction_report['rows'],
    'submission_customers': submission_report['rows'],
    'transaction_date_range': [transaction_report['date_min'], transaction_report['date_max']],
    'invalid_or_unknown_rows': sum(value for key, value in transaction_report.items() if key in {'malformed_customer_ids', 'malformed_article_ids', 'invalid_dates', 'invalid_prices', 'invalid_sales_channels', 'unknown_customers', 'unknown_articles'}),
    'output_dir': str(OUTPUT_DIR),
}
print(json.dumps(summary, ensure_ascii=False, indent=2))
print(json.dumps(quality_report['artifacts'], ensure_ascii=False, indent=2))